In [1]:

import os
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoTokenizer, AutoModel

from datasets import load_dataset

import datasets, sys
print(f"versão dataset: {datasets.__version__}")

# Um limite amostral para os prints não ficarem enormes
lim_print = 5

/Users/hissapinto/Library/CloudStorage/OneDrive-Pessoal/_CC/IC/doc-ranker/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


versão dataset: 2.18.0


#### Função `average_pool`

<span style="font-size: 0.85em">

Reduz a saída token-a-token do modelo a um único vetor por sentença, ignorando os tokens de padding.

#### Parâmetros

**`last_hidden_states`** — tensor de saída da última camada do modelo, com shape `(batch, seq_len, hidden)`.

**`attention_mask`** — tensor que define quais tokens são reais (valor `1`) e quais devem ser ignorados (valor `0`). Shape `(batch, seq_len)`: uma linha por sequência do batch, uma posição por token.

#### Conceitos

**Tensor** — array multidimensional de números.

**Shape** — tupla com um valor por dimensão, indicando o tamanho de cada uma:

| Dimensões | Exemplo | Shape |
|---|---|---|
| 0 (escalar) | `3.14` | `()` |
| 1 (vetor) | `[1, 2, 3]` | `(3,)` |
| 2 (matriz) | `[[1, 2], [3, 4]]` | `(2, 2)` |
| 3 (cubo) | `[[[1,2],[3,4]], [[5,6],[7,8]]]` | `(2, 2, 2)` |
| N | sem nome próprio, mesma ideia | — |

#### Como funciona

Cada token vira um vetor de tamanho `hidden` (768, 1024 — depende do modelo), e a média é tirada **posição a posição**: a dimensão 0 do vetor final é a média das dimensões 0 de todos os tokens, e assim por diante.

O resultado tem exatamente o mesmo tamanho de um vetor de token individual, mas agora representa a sentença inteira.

</span>

In [5]:
def average_pool(last_hidden_states: Tensor, attention_mask: Tensor) -> Tensor:
    
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

#### Carregamento do modelo

<span style="font-size: 0.85em">

```python
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')
```

**`AutoTokenizer`** — converte texto em IDs numéricos. Devolve `input_ids` (os tokens) e `attention_mask` (quais posições são reais).

**`AutoModel`** — a rede em si, sem cabeça de tarefa. Devolve `last_hidden_state` com shape `(batch, seq_len, 768)`.

**`Auto*`** — classes genéricas: leem a configuração do repositório e instanciam a arquitetura correta (aqui, XLM-RoBERTa) sem que você precise nomeá-la.

**`from_pretrained(...)`** — baixa os pesos do Hugging Face Hub na primeira execução e os guarda em cache local; nas seguintes, carrega do disco.

**`multilingual-e5-base`** — modelo de embeddings multilíngue (inclui português), `hidden = 768`. Usa mean pooling, daí a `average_pool`.
</span>

In [6]:
tokenizer = AutoTokenizer.from_pretrained('intfloat/multilingual-e5-base')
model = AutoModel.from_pretrained('intfloat/multilingual-e5-base')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8031.59it/s]


#### Carregamento e tokenização dos documentos

<span style="font-size: 0.85em">

#### `load_dataset`

**`'unicamp-dl/mmarco'`** — versão multilíngue do MS MARCO, corpus padrão de recuperação de documentos.

**`'collection-portuguese'`** — a *config*: seleciona a coleção de passagens em português. É o acervo de textos, não muda entre treino e teste.

**`streaming=True`** — não baixa o dataset inteiro (são milhões de passagens); os exemplos chegam sob demanda. O retorno é um `IterableDataset`, que não aceita indexação por posição.

**`trust_remote_code=True`** — autoriza a execução do script `mmarco.py` do repositório. Exige `datasets < 4.0`; nas versões novas, loading scripts foram removidos.

#### Amostragem

**`.take(100)`** — pega os 100 primeiros exemplos do stream. O `list(...)` materializa o iterador em memória.

**`item['text']`** — extrai só o texto de cada registro, descartando o `id`.

#### `tokenizer`

**`max_length=512`** — limite de tokens por sequência.

**`truncation=True`** — corta o que passar desse limite.

**`padding=True`** — completa as sequências curtas até o comprimento da maior do batch, para formar um tensor retangular. É o que cria os paddings que a `average_pool` precisa ignorar.

**`return_tensors='pt'`** — devolve tensores PyTorch em vez de listas.

O resultado é um dicionário com `input_ids` e `attention_mask`, ambos com shape `(100, seq_len)`.

</span>

In [7]:
# Passa os parâmetros do dataset a ser buscado
dataset_docs = load_dataset('unicamp-dl/mmarco', 'collection-portuguese', streaming=True, trust_remote_code=True)

# Pega só os primeiros 100 exemplos
docs = list(dataset_docs['collection'].take(100))
print(f"Exemplo docs:\n{docs[0]}\n")

# Separa o texto do id e labels
collection_textos = [f"{item['text']}" for item in docs]
print(f"Exemplo texto:\n{collection_textos[0]}\n")

# transforma de texto para token e depois em id e attention mask
docs_batch = tokenizer(collection_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')
print(f"ids:\n{docs_batch['input_ids'][:lim_print, :lim_print]}\n")
print(f"máscara de atenção:\n{docs_batch['attention_mask'][:lim_print, :lim_print]}")

Repo card metadata block was not found. Setting CardData to empty.


Exemplo docs:
{'id': 0, 'text': 'A presença de comunicação entre mentes científicas foi tão importante para o sucesso do Projeto Manhattan quanto o intelecto científico. A única nuvem que paira sobre a impressionante conquista dos pesquisadores e engenheiros atômicos é o que seu sucesso realmente significou; centenas de milhares de vidas inocentes destruídas.'}

Exemplo texto:
A presença de comunicação entre mentes científicas foi tão importante para o sucesso do Projeto Manhattan quanto o intelecto científico. A única nuvem que paira sobre a impressionante conquista dos pesquisadores e engenheiros atômicos é o que seu sucesso realmente significou; centenas de milhares de vidas inocentes destruídas.

ids:
tensor([[     0,     62,  83541,      8,  98146],
        [     0,    180, 121926, 147037,     28],
        [     0,    357,    433,    846,   1028],
        [     0,    180, 121926, 147037,   1615],
        [     0,    493,  50364,      8,   2124]])

máscara de atenção:
tensor([[1, 1

#### Carregamento e tokenização das consultas

<span style="font-size: 0.85em">

**`'queries-portuguese'`** — config com as consultas do MS MARCO, as queries são as perguntas para as "passage".

**`['train']`** — acessa os dados definidos para treino. Datasets costumam vir separados em train, validation e test para que treino e avaliação usem dados distintos.

**Prefixo `"query: "`** — obrigatório no E5, e distinto do `"passage: "` usado nos documentos. É assim que o modelo diferencia os dois papéis; sem isso o ranking degrada.

**`max_length=512`** — herdado do batch de documentos, mas consultas são curtas (poucas palavras). Como `padding=True` preenche até a maior sequência *do batch*, e não até 512, não há desperdício.

O resultado tem a mesma estrutura do `docs_batch`: `input_ids` e `attention_mask` com shape `(100, seq_len)`.

</span>

In [8]:
# Passa os parâmetros do dataset a ser buscado
dataset_queries = load_dataset('unicamp-dl/mmarco', 'queries-portuguese', streaming=True, trust_remote_code=True)

# Pega só os primeiros 100 exemplos
queries = list(dataset_queries['train'].take(100))
print(f"Exemplo queries:\n{queries[1]}\n")

# Separa o texto.
queries_textos = [f"{item['text']}" for item in queries]
print(f"Exemplo query:\n{queries_textos[1]}\n")

# transforma de texto para token e depois em id e attention mask
queries_batch = tokenizer(queries_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')
print(f"ids:\n{queries_batch['input_ids'][:lim_print, :lim_print]}\n")
print(f"máscara de atenção:\n{queries_batch['attention_mask'][:lim_print, :lim_print]}")

Repo card metadata block was not found. Setting CardData to empty.


Exemplo queries:
{'id': 634306, 'text': 'o que significa bem móvel no histórico de crédito'}

Exemplo query:
o que significa bem móvel no histórico de crédito

ids:
tensor([[     0,  48106, 132038,      2,      1],
        [     0,     36,     41,  12330,   5289],
        [     0,   7002,   1715,     36,   3189],
        [     0,  47739,   7685,      8,  33780],
        [     0,     36,     41,    393,  14543]])

máscara de atenção:
tensor([[1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1]])


#### Geração dos embeddings contextuais das consultas

<span style="font-size: 0.85em">

**`model(**queries_batch)`** — o `**` desempacota o dicionário do tokenizer; equivale a `model(input_ids=..., attention_mask=...)`.

**`queries_outputs`** — objeto `BaseModelOutput`, acessível por atributo. O campo relevante é `last_hidden_state`, com shape `(100, seq_len, 768)`: um vetor por token.

**`average_pool(...)`** — reduz a `(100, 768)`: um vetor por consulta. A máscara é passada de novo porque o modelo não a devolve na saída — ele retorna vetores para todas as posições, inclusive os paddings.

**`F.normalize(..., p=2, dim=1)`** — normalização L2 ao longo da dimensão 1 (a das 768 features): cada vetor passa a ter comprimento 1. Com isso o produto interno vira cosseno, e a similaridade depende só da direção, não da magnitude.

**`(100,768)`** — 100 foi o número escolhido de textos(itens no batch) e 768 é o tamanho de cada vetor da arquitetura **`multilingual-e5-base`**. e o **`seq_len`**. é a quantidade de tokens de cada sequência do batch.

#### Fluxo dos shapes

```
last_hidden_state    (100, seq_len, 768)   um vetor por token
      ↓ average_pool
queries_embeddings   (100, 768)            um vetor por consulta
      ↓ F.normalize
queries_embeddings   (100, 768)            mesmo shape, comprimento 1
```

</span>

In [9]:
# Calcula os embeddings contextuais de cada posição do batch
queries_outputs = model(**queries_batch)
print(f"Exemplo de um embedding contextual de uma query:\n{queries_outputs.last_hidden_state[0][:lim_print, :lim_print]}\n")

# Faz a média de todos os embeddings em um só tensor -> contexto geral da query
queries_embeddings = average_pool(queries_outputs.last_hidden_state, queries_batch['attention_mask'])
print(f"Exemplo média de todos os embeddings de uma query:\n{queries_embeddings[0][:lim_print]}\n")

# Normaliza os vetores (Normalização L2) para que todos tenham um "comprimento" (raiz da soma dos quadrados) igual a 1
# p=2 pois é a normalização L2 = euclidiana.
queries_embeddings = F.normalize(queries_embeddings, p=2, dim=1)
print(f"Exemplo de embedding normalizada:\n{queries_embeddings[0][:lim_print]}")

Exemplo de um embedding contextual de uma query:
tensor([[-0.0015,  0.3886, -0.0278, -0.0125,  0.0957],
        [-0.1981,  0.3106,  0.0151,  0.1930,  0.4064],
        [-0.1087,  0.3781, -0.0808,  0.2984,  0.0751],
        [-0.1120,  0.4233,  0.0679,  0.0373,  0.4476],
        [ 0.0205,  0.3841,  0.0346,  0.4197,  0.5480]],
       grad_fn=<SliceBackward0>)

Exemplo média de todos os embeddings de uma query:
tensor([-0.1051,  0.3752, -0.0064,  0.1290,  0.2562], grad_fn=<SliceBackward0>)

Exemplo de embedding normalizada:
tensor([-0.0064,  0.0229, -0.0004,  0.0079,  0.0156], grad_fn=<SliceBackward0>)


### Geração dos embeddings dos documentos

<span style="font-size: 0.85em">

Mesmo processo aplicado às consultas, agora sobre as passagens da coleção — o **mesmo modelo** e a **mesma normalização**, condição para que os dois conjuntos de vetores sejam comparáveis.

**`model(**docs_batch)`** — `last_hidden_state` com shape `(100, seq_len, 768)`. Aqui o `seq_len` é bem maior que o das consultas: passagens têm dezenas de tokens, consultas têm poucos. A diferença some no pooling.

**`average_pool(...)`** — reduz a `(100, 768)`: um vetor por passagem.

**`F.normalize(..., p=2, dim=1)`** — comprimento 1 em cada vetor, ele diz que a normalização é feita ao longo das 768 features, uma linha de cada vez. Se fosse dim=0, cada coluna teria comprimento 1 — o que misturaria as 100 consultas entre si e não faria sentido nenhum..

</span>

In [10]:
# Calcula os embeddings contextuais de cada posição do batch
docs_outputs = model(**docs_batch)
print(f"Exemplo de um embedding contextual de um texto:\n{docs_outputs.last_hidden_state[0][:lim_print, :lim_print]}\n")

# Faz a média de todos os embeddings em um só tensor -> contexto geral do texto
docs_embeddings = average_pool(docs_outputs.last_hidden_state, docs_batch['attention_mask'])
print(f"Exemplo média de todos os embeddings de um texto:\n{queries_embeddings[0][:lim_print]}\n")

# Normaliza os vetores (Normalização L2) para que todos tenham um "comprimento" (raiz da soma dos quadrados) igual a 1
docs_embeddings = F.normalize(docs_embeddings, p=2, dim=1)
print(f"Exemplo de embedding normalizada:\n{docs_embeddings[0][:lim_print]}")

Exemplo de um embedding contextual de um texto:
tensor([[-0.6357,  1.1537,  0.3374,  0.5907,  0.5854],
        [-0.6798,  1.4169, -0.0771,  0.6531,  0.9560],
        [-0.6899,  1.3160,  0.1435,  0.7221,  1.1289],
        [-0.4698,  1.1369, -0.0626,  0.8005,  0.8721],
        [-0.4703,  1.3623,  0.2218,  0.7894,  0.9574]],
       grad_fn=<SliceBackward0>)

Exemplo média de todos os embeddings de um texto:
tensor([-0.0064,  0.0229, -0.0004,  0.0079,  0.0156], grad_fn=<SliceBackward0>)

Exemplo de embedding normalizada:
tensor([-0.0288,  0.0693,  0.0060,  0.0407,  0.0435], grad_fn=<SliceBackward0>)


#### Cálculo dos scores

<span style="font-size: 0.85em">

**Score** — número que mede o quanto uma consulta e uma passagem se parecem. Como os vetores estão normalizados, o produto interno equivale ao cosseno do ângulo entre eles.

**`@`** — multiplicação de matrizes. `(100, 768) @ (768, 100)` → `(100, 100)`. A transposta `.T` alinha as dimensões internas: o 768 de um encosta no 768 do outro.

**`scores[i][j]`** — similaridade entre a consulta `i` e a passagem `j`. A linha `i` inteira são os 100 scores daquela consulta contra toda a coleção — o ranking bruto, ainda sem ordenar.

**`.tolist()`** — converte para lista Python. Perde as operações do tensor (`topk`, `argsort`); vale manter em PyTorch enquanto houver ranking a fazer.

#### Lendo os valores

A escala vai de -1 a 1, mas no E5 os scores se concentram entre 0.7 e 0.9 mesmo para pares sem relação. O valor absoluto diz pouco — **o que importa é a comparação dentro da mesma linha**, entre as passagens candidatas de uma mesma consulta.

Por isso a avaliação em recuperação usa métricas de ordenação (MRR@10, nDCG@10), que olham a posição da resposta correta, não o valor do score.

</span>

In [12]:
# scores são os números que mede o quanto uma consulta e uma passagem se parecem
scores = queries_embeddings @ docs_embeddings.T   # (100, 100)
# os 100 scores da primeira consulta contra cada uma das 100 passagens da coleção.
# O valor na posição j é a similaridade entre a query 0 e a passagem j
print(f"Exemplo score:\n{scores[0]}\n")

# Converte a matriz de tensores para uma lista do Python
scores_list = scores.tolist()


Exemplo score:
tensor([0.7511, 0.7457, 0.7606, 0.7470, 0.7548, 0.7603, 0.7394, 0.7561, 0.7561,
        0.7718, 0.7664, 0.7808, 0.7324, 0.7172, 0.7683, 0.7600, 0.7588, 0.7624,
        0.7594, 0.7470, 0.7556, 0.7633, 0.7680, 0.7578, 0.7463, 0.7405, 0.7457,
        0.7441, 0.7295, 0.7215, 0.7422, 0.7332, 0.7560, 0.7241, 0.7452, 0.7286,
        0.7443, 0.7291, 0.7802, 0.7547, 0.7582, 0.7801, 0.7413, 0.7398, 0.7726,
        0.7367, 0.7397, 0.7340, 0.7511, 0.7498, 0.7507, 0.7404, 0.7624, 0.7474,
        0.7632, 0.7649, 0.7442, 0.7664, 0.7482, 0.7474, 0.7547, 0.7451, 0.7719,
        0.7510, 0.7563, 0.7629, 0.7720, 0.7563, 0.7469, 0.7555, 0.7649, 0.7424,
        0.7687, 0.7318, 0.7583, 0.7598, 0.7432, 0.7730, 0.7366, 0.7572, 0.7743,
        0.7377, 0.7819, 0.7647, 0.7599, 0.7356, 0.7657, 0.7685, 0.7684, 0.7350,
        0.7507, 0.7371, 0.7418, 0.7734, 0.7521, 0.7402, 0.7541, 0.7498, 0.7327,
        0.7477], grad_fn=<SelectBackward0>)



### Ranqueamento e exibição dos resultados

<span style="font-size: 0.85em">

**`enumerate(queries_textos)`** — percorre as consultas mantendo o índice `i`, usado para pegar a linha de scores correspondente.

**`zip(scores_list[i], collection_textos)`** — Ele combina listas paralelas em pares. No contexto atual, ele emparelha cada score com sua passagem, formando uma tupla. A correspondência funciona porque a ordem de `collection_textos` é a mesma usada na tokenização.

**`sorted(..., key=lambda x: x[0], reverse=True)`** — ordena pelos scores (`x[0]` é o primeiro item de cada par), do maior para o menor.

**`[:1]`** — mantém só o primeiro colocado. Trocar por `[:3]` ou `[:5]` mostra mais candidatos.

**`replace('\n', ' ')`** — achata quebras de linha internas para não bagunçar a saída.

#### Pontos de atenção

**O limiar `> 85` nunca dispara.** Os scores estão na escala de 0 a 1 (~0.75). O valor 85 pressupõe a multiplicação por 100 do exemplo oficial do E5, que não foi aplicada. Destacar por posição (`rank == 1`) é mais robusto que um limiar fixo. No arquivo e5_test_2.py funciona corretamente.

**`{score:.1f}` esconde a diferença.** Com uma casa, `0.7819` e `0.7802` viram ambos `0.8`. Use `:.4f`.

In [13]:
# Ranqueamento
# Loop para imprimir os resultados organizados
for i, query in enumerate(queries_textos):
    print(f"\nPERGUNTA: '{query}'")
    
    # 1. Juntamos as notas (scores) com seus respectivos textos
    resultados = list(zip(scores_list[i], collection_textos))
    
    # 2. Ordenamos a lista baseada na nota (x[0]), do maior para o menor (reverse=True)
    resultados_ordenados = sorted(resultados, key=lambda x: x[0], reverse=True)
    
    # 3. Imprimimos os resultados já ordenados
    for score, passage in resultados_ordenados[:1]:
        # Cortamos o texto em 60 caracteres
        texto_curto = passage[:60].replace('\n', ' ') + "..." 
        
        # Destaca visualmente se o score for maior que 85
        destaque = "✅" if score > 85 else "  "
        
        print(f"MELHOR RESPOSTA: {destaque} Score: {score:.4f} | Texto: {texto_curto}")
        print("-" * 100)


PERGUNTA: 'definir extremo'
MELHOR RESPOSTA:    Score: 0.7819 | Texto: Para os ossos individuais, consulte Rib. Para costelas de an...
----------------------------------------------------------------------------------------------------

PERGUNTA: 'o que significa bem móvel no histórico de crédito'
MELHOR RESPOSTA:    Score: 0.8100 | Texto: Ele é bacharel em História Legal e Constitucional Americana ...
----------------------------------------------------------------------------------------------------

PERGUNTA: 'qual foi o grande salto para a frente cerebralmente'
MELHOR RESPOSTA:    Score: 0.8033 | Texto: A presença de comunicação entre mentes científicas foi tão i...
----------------------------------------------------------------------------------------------------

PERGUNTA: 'fixadores de tatuagem quanto custa'
MELHOR RESPOSTA:    Score: 0.8001 | Texto: Custo do tratamento médico na Costa Rica. A seguir estão as ...
----------------------------------------------------------------

### Avaliação: MRR@10

<span style="font-size: 0.85em">

**MRR (Mean Reciprocal Rank)** — média do inverso da posição em que o documento relevante aparece no ranking, calculada sobre todas as consultas.

Para uma consulta `i`, o *reciprocal rank* é `1 / rank_i`, onde `rank_i` é a posição (1-indexada) do documento correto:

| Posição do documento certo | Reciprocal rank |
|---|---|
| 1º | 1.000 |
| 2º | 0.500 |
| 3º | 0.333 |
| 10º | 0.100 |
| fora do top-10 | 0.000 |

**MRR@10** — corta o ranking nos 10 primeiros (`k = 10`): só conta como acerto se o documento certo estiver no top-10. O MRR final é a média desses valores entre todas as consultas avaliadas.

#### Por que MRR e não nDCG aqui

O mMARCO (como o MS MARCO original) traz **um único documento relevante por consulta**, com relevância **binária** (certo/errado). Nesse cenário o MRR@10 é a métrica padrão da literatura — é o número reportado no leaderboard oficial do MS MARCO — e equivale a um nDCG@10 sobre o mesmo gabarito, sem a complexidade extra de graus de relevância que aqui não existem. O nDCG@k só ganha vantagem com relevância graduada (ex.: TREC DL, notas 0–3) ou vários relevantes por consulta.

#### O que a avaliação exige

As 100 consultas e 100 passagens amostradas acima vêm de *streams independentes*: nada garante (e na prática quase nunca acontece) que a passagem certa de uma consulta esteja entre as 100 amostradas — com um recorte de 100 em ~8,8 milhões, a chance é de ~0,001%. Medir MRR@10 direto sobre elas daria 0 sempre, sem significado.

Por isso os passos abaixo:

1. usar o split **`dev`** das consultas — o único com gabarito de relevância (*qrels*) publicado;
2. buscar, na coleção completa, a passagem realmente relevante de cada consulta avaliada;
3. montar um **pool de candidatos** com essas passagens certas + as 100 já amostradas como distratoras, e ranquear dentro desse pool.

Isso é uma avaliação em **pool fechado** — não é recuperação sobre a coleção inteira, que exigiria indexar os ~8,8 milhões de vetores — mas garante que o gabarito esteja presente, que é a condição para a métrica significar alguma coisa.

</span>

In [14]:
def mrr_at_k(scores: Tensor, doc_ids: list[int], relevant_ids: dict[int, int], k: int = 10) -> float:
    """
    Calcula o MRR@k (Mean Reciprocal Rank) de um conjunto de consultas.

    scores       -- matriz (n_queries, n_docs): scores[i][j] é a similaridade
                    entre a consulta i e o documento da coluna j
    doc_ids      -- id oficial de cada coluna de `scores`, na mesma ordem
    relevant_ids -- {índice da consulta em `scores`: id do documento relevante}
    k            -- profundidade de corte do ranking (k=10 -> MRR@10)
    """
    # mapeia id do documento -> coluna correspondente em `scores`
    doc_id_to_col = {doc_id: col for col, doc_id in enumerate(doc_ids)}

    reciprocal_ranks = []

    for query_idx, relevant_id in relevant_ids.items():
        # índices das colunas ordenados do mais para o menos similar, cortado em k
        ranking = torch.argsort(scores[query_idx], descending=True)[:k]

        relevant_col = doc_id_to_col.get(relevant_id)

        if relevant_col is None:
            # o documento relevante nem está no pool avaliado
            reciprocal_ranks.append(0.0)
            continue

        # posição do documento relevante dentro do top-k (tensor vazio se ficou de fora)
        hits = (ranking == relevant_col).nonzero()

        if hits.numel() == 0:
            reciprocal_ranks.append(0.0)
        else:
            rank_position = hits.item() + 1   # +1 porque o índice do tensor é 0-indexado
            reciprocal_ranks.append(1.0 / rank_position)

    return sum(reciprocal_ranks) / len(reciprocal_ranks)

#### Carregamento do gabarito (qrels) e das consultas de avaliação

<span style="font-size: 0.85em">

**qrels** — *query relevance judgments*, o gabarito da tarefa: para cada consulta, qual documento é considerado relevante. Sem ele não existe métrica de ranking.

**`data/qrels.dev.small.tsv`** — arquivo oficial do mMARCO, no formato `query_id \t 0 \t passage_id \t 1`. As colunas 0 e 1 são herança do formato TREC (número da iteração e grau de relevância); no MS MARCO só interessam a 1ª e a 3ª.

**`hf_hub_download`** — baixa um arquivo avulso do repositório, sem passar pelo *loading script*. É preciso aqui porque o `mmarco.py` não expõe os qrels como config do `load_dataset`. O arquivo tem ~143 KB e fica em cache local.

**Split `dev`** — corresponde a `portuguese_queries.dev.small.tsv`, exatamente o conjunto coberto pelo qrels acima (as 6.980 consultas do *small dev set* oficial). É por isso que os dois são carregados juntos.

**Por que não o split `train`** — o mMARCO não publica qrels para o treino, só para o dev. As `queries_textos` usadas mais acima servem para demonstrar o pipeline de embeddings, não para avaliar.

**`N_EVAL_QUERIES`** — quantidade de consultas avaliadas. Mantido baixo de propósito: cada uma exige localizar sua passagem relevante na coleção completa (próxima célula), que é a parte cara do processo.

</span>

In [15]:
from huggingface_hub import hf_hub_download

# Quantas consultas do dev set serão avaliadas.
# Vale usar um valor alto: o custo dominante é a varredura da coleção (célula adiante),
# que percorre quase o arquivo inteiro de qualquer jeito. Avaliar 100 consultas custa
# praticamente o mesmo que avaliar 10, e dá uma métrica muito menos ruidosa.
N_EVAL_QUERIES = 100

# Baixa o gabarito oficial (~143 KB, fica em cache)
qrels_path = hf_hub_download(
    repo_id='unicamp-dl/mmarco',
    repo_type='dataset',
    filename='data/qrels.dev.small.tsv',
)

# Lê o qrels como {query_id: id da passagem relevante}
qrels = {}
with open(qrels_path, encoding='utf-8') as f:
    for line in f:
        query_id, _, passage_id, _ = line.rstrip('\n').split('\t')
        qrels[int(query_id)] = int(passage_id)

print(f"qrels carregado: {len(qrels)} consultas com gabarito\n")

# Carrega as primeiras consultas do split 'dev' (o mesmo conjunto coberto pelo qrels)
dataset_queries_dev = load_dataset('unicamp-dl/mmarco', 'queries-portuguese', streaming=True, trust_remote_code=True)
eval_queries = list(dataset_queries_dev['dev'].take(N_EVAL_QUERIES))
eval_queries_textos = [item['text'] for item in eval_queries]
print(f"Exemplo consulta de avaliação:\n{eval_queries[0]}\n")

# {índice da consulta na lista: id da passagem relevante} -- só as que têm gabarito
relevant_ids = {i: qrels[item['id']] for i, item in enumerate(eval_queries) if item['id'] in qrels}
print(f"{len(relevant_ids)}/{len(eval_queries)} consultas de avaliação com gabarito encontrado")

qrels carregado: 6980 consultas com gabarito



Repo card metadata block was not found. Setting CardData to empty.


Exemplo consulta de avaliação:
{'id': 1048585, 'text': 'o que é irmão de paula deen'}

100/100 consultas de avaliação com gabarito encontrado


#### Busca das passagens relevantes na coleção completa

<span style="font-size: 0.85em">

O gabarito aponta ids de passagem de toda a coleção (~8,8 milhões), não do recorte de 100 amostrado lá em cima. Sem trazer essas passagens para o pool, a métrica seria zero por construção.

**Por que uma passada só** — em modo `streaming=True` não há acesso por posição: chegar ao id *n* significa percorrer os *n* primeiros registros. Buscar uma passagem por vez custaria uma varredura por consulta. O laço abaixo faz **uma varredura**, recolhendo todos os ids de `target_ids` no caminho.

**O custo é praticamente fixo** — os ids do gabarito estão espalhados por toda a coleção, então o maior deles quase sempre cai perto do fim do arquivo: uma amostra qualquer de consultas exige percorrer ~90–100% dos ~3,4 GB. Diminuir `N_EVAL_QUERIES` **não** acelera de forma relevante — mas aumentá-lo é quase de graça. Daí o valor 100 na célula anterior: mesma varredura, métrica bem menos ruidosa.

**Cache** — as passagens encontradas são gravadas em `passagens_relevantes_cache.json`, ao lado do notebook. Da segunda execução em diante a varredura é pulada, desde que `N_EVAL_QUERIES` não aumente. Apague o arquivo para forçar nova busca.

**`break` antecipado** — encerra assim que a última passagem procurada aparece, evitando percorrer o resto do arquivo à toa.

**Atenção ao tempo:** na primeira execução, conte com alguns minutos de streaming (não é download permanente — os bytes são lidos e descartados).

</span>

In [ ]:
import json
import time

CACHE_PATH = Path('../data/passagens_relevantes_cache.json')

# Ids das passagens que precisam estar no pool de avaliação
target_ids = set(relevant_ids.values())

# Reaproveita o que já foi encontrado em execuções anteriores
cache = {}
if CACHE_PATH.exists():
    cache = {int(k): v for k, v in json.loads(CACHE_PATH.read_text(encoding='utf-8')).items()}

found_passages = {doc_id: cache[doc_id] for doc_id in target_ids if doc_id in cache}
faltando = target_ids - found_passages.keys()
print(f"{len(found_passages)}/{len(target_ids)} passagens vieram do cache")

# Só varre a coleção se ainda faltar alguma
if faltando:
    print(f"Procurando {len(faltando)} passagens (maior id: {max(faltando)})... pode demorar alguns minutos")
    inicio = time.time()

    dataset_docs_full = load_dataset('unicamp-dl/mmarco', 'collection-portuguese', streaming=True, trust_remote_code=True)
    for item in dataset_docs_full['collection']:
        if item['id'] in faltando:
            found_passages[item['id']] = item['text']

            # para assim que todas forem encontradas
            if len(found_passages) == len(target_ids):
                break

    print(f"Varredura concluída em {time.time() - inicio:.0f}s")

    # Atualiza o cache com tudo que foi encontrado
    cache.update(found_passages)
    CACHE_PATH.write_text(json.dumps(cache, ensure_ascii=False), encoding='utf-8')

print(f"\n{len(found_passages)}/{len(target_ids)} passagens relevantes disponíveis\n")

exemplo_id = next(iter(found_passages))
print(f"Exemplo de passagem relevante (id {exemplo_id}):\n{found_passages[exemplo_id][:200]}...")

0/100 passagens vieram do cache
Procurando 100 passagens (maior id: 7963717)... pode demorar alguns minutos


Repo card metadata block was not found. Setting CardData to empty.


Varredura concluída em 686s

100/100 passagens relevantes disponíveis

Exemplo de passagem relevante (id 22297):
Wonderful Tonight foi escrito por Eric Clapton. Foi incluído no álbum Slowhand de Clapton de 1977 e lançado como single no ano seguinte. Em 1988, Clapton apareceu no concerto Nelson Mandela 70th Birth...


#### Pool de candidatos e scores da avaliação

<span style="font-size: 0.85em">

O pool contra o qual cada consulta é ranqueada junta:

- as **100 passagens já amostradas** (`collection_textos`), que funcionam como distratoras;
- as **passagens relevantes** recuperadas na célula anterior, uma por consulta avaliada.

**Reaproveitamento** — `docs_embeddings` já tem as 100 distratoras codificadas e normalizadas; só as passagens novas e as consultas de avaliação passam pelo modelo. O pipeline é idêntico ao usado antes (`tokenizer` → `model` → `average_pool` → `F.normalize`), condição para que os vetores sejam comparáveis.

**`torch.cat(..., dim=0)`** — empilha os dois blocos de embeddings ao longo da dimensão dos documentos: `(100, 768)` + `(n_novas, 768)` → `(100 + n_novas, 768)`.

**`eval_pool_ids`** — a lista de ids na mesma ordem das linhas do pool. É o que permite ao `mrr_at_k` traduzir "id do gabarito" em "coluna de `eval_scores`".

**`torch.no_grad()`** — desliga o rastreamento de gradientes. Aqui é só inferência, não há treino; economiza memória e tempo.

#### Shapes

```
eval_queries_embeddings   (N_EVAL_QUERIES, 768)
eval_pool_embeddings      (100 + n_novas, 768)
      ↓ @ .T
eval_scores               (N_EVAL_QUERIES, 100 + n_novas)
```

</span>

In [17]:
# Ids do pool, na mesma ordem das linhas dos embeddings: 100 distratoras + relevantes encontradas
eval_pool_ids = [item['id'] for item in docs] + list(found_passages.keys())

# Embeddings das passagens relevantes (as 100 distratoras já estão em docs_embeddings)
novas_passagens = list(found_passages.values())
novas_batch = tokenizer(novas_passagens, max_length=512, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    novas_outputs = model(**novas_batch)

novas_embeddings = average_pool(novas_outputs.last_hidden_state, novas_batch['attention_mask'])
novas_embeddings = F.normalize(novas_embeddings, p=2, dim=1)

# Junta os dois blocos em um único pool
eval_pool_embeddings = torch.cat([docs_embeddings, novas_embeddings], dim=0)
print(f"Pool de avaliação: {eval_pool_embeddings.shape}")

# Embeddings das consultas de avaliação, mesmo pipeline usado nas consultas de treino
eval_queries_batch = tokenizer(eval_queries_textos, max_length=512, padding=True, truncation=True, return_tensors='pt')

with torch.no_grad():
    eval_queries_outputs = model(**eval_queries_batch)

eval_queries_embeddings = average_pool(eval_queries_outputs.last_hidden_state, eval_queries_batch['attention_mask'])
eval_queries_embeddings = F.normalize(eval_queries_embeddings, p=2, dim=1)
print(f"Consultas de avaliação: {eval_queries_embeddings.shape}\n")

# Similaridade de cada consulta contra cada documento do pool
eval_scores = eval_queries_embeddings @ eval_pool_embeddings.T
print(f"Matriz de scores: {eval_scores.shape}")

Pool de avaliação: torch.Size([200, 768])
Consultas de avaliação: torch.Size([100, 768])

Matriz de scores: torch.Size([100, 200])


#### Resultado do MRR@10

<span style="font-size: 0.85em">

A célula abaixo detalha, consulta a consulta, em que posição o documento certo ficou, e fecha com o MRR@10 — a média dos *reciprocal ranks*.

#### Como ler o número

| MRR@10 | Interpretação |
|---|---|
| 1.00 | o documento certo é sempre o 1º colocado |
| ~0.50 | em média, o certo aparece por volta da 2ª posição |
| ~0.30 | em torno da 3ª posição — patamar típico de modelos densos no MS MARCO completo |
| 0.00 | o certo nunca aparece no top-10 |

**Cuidado na comparação com a literatura:** o MRR@10 oficial do MS MARCO é medido contra **~8,8 milhões** de passagens, e os valores publicados (E5 e similares ficam na casa de 0,30–0,40) refletem essa dificuldade. Aqui o pool tem ~200 documentos (100 distratoras + as relevantes das consultas avaliadas), então o número sai **bem mais otimista** — serve para validar que o pipeline funciona e para comparar variações entre si (modelo A × modelo B, com ou sem prefixo `query:`), não para ser confrontado com o leaderboard.

**Um viés a ter em mente:** as passagens relevantes das *outras* consultas entram como distratoras, o que é bom — são textos reais e temáticos, não ruído. Ainda assim, com ~200 candidatos a tarefa é ordens de magnitude mais fácil que a real.

**Para aproximar do cenário real**, aumente o pool de distratoras: troque o `.take(100)` da amostragem de documentos por um valor maior e recalcule `docs_embeddings`. Quanto maior o pool, mais difícil — e mais informativo — fica o MRR@10.

</span>

In [18]:
K = 10
lim_detalhe = 10   # quantas consultas detalhar no print (a métrica usa todas)

doc_id_to_col = {doc_id: col for col, doc_id in enumerate(eval_pool_ids)}

# Detalhamento consulta a consulta
for query_idx, relevant_id in list(relevant_ids.items())[:lim_detalhe]:
    ranking = torch.argsort(eval_scores[query_idx], descending=True)[:K]
    hits = (ranking == doc_id_to_col[relevant_id]).nonzero()

    posicao = f"{hits.item() + 1}º" if hits.numel() else f"fora do top-{K}"
    rr = 1.0 / (hits.item() + 1) if hits.numel() else 0.0
    destaque = "✅" if hits.numel() and hits.item() == 0 else "  "

    print(f"{destaque} CONSULTA: '{eval_queries_textos[query_idx]}'")
    print(f"   passagem relevante (id {relevant_id}): {posicao}  |  RR = {rr:.4f}")
    print("-" * 100)

# Métrica final, sobre todas as consultas avaliadas
mrr10 = mrr_at_k(eval_scores, eval_pool_ids, relevant_ids, k=K)

print(f"\nMRR@{K} = {mrr10:.4f}")
print(f"({len(relevant_ids)} consultas avaliadas, pool de {len(eval_pool_ids)} documentos)")

✅ CONSULTA: 'o que é irmão de paula deen'
   passagem relevante (id 7187158): 1º  |  RR = 1.0000
----------------------------------------------------------------------------------------------------
✅ CONSULTA: ' Receptor de andrógeno define'
   passagem relevante (id 4339068): 1º  |  RR = 1.0000
----------------------------------------------------------------------------------------------------
✅ CONSULTA: 'tratamento de dores de cabeça tensionais sem medicação'
   passagem relevante (id 740662): 1º  |  RR = 1.0000
----------------------------------------------------------------------------------------------------
✅ CONSULTA: 'o que é paranóico sc'
   passagem relevante (id 7187023): 1º  |  RR = 1.0000
----------------------------------------------------------------------------------------------------
✅ CONSULTA: 'tratamento de varizes nas pernas'
   passagem relevante (id 7573435): 1º  |  RR = 1.0000
-------------------------------------------------------------------------------------